# SIH26142 — Deep Super-Resolution Fine-Tuning (Google Colab)
### Sentinel-2 4× Super-Resolution | Real-ESRGAN (RRDBNet Backbone) + Joint Spectral Modeling

**What this notebook does:**
1. Verifies T4 GPU and sets up environment with `rasterio`, `torch`, `basicsr`, `realesrgan`
2. Clones the project from GitHub and reloads updated `src` modules
3. Verifies / generates synthetic training pairs from Sentinel-2 10m GeoTIFFs
4. Runs unified fine-tuning (`src/train.py`) with **L1 + VGG16 Perceptual + SAM + Sobel Edge + FFT Frequency + Multi-Scale Pyramid** loss
5. Supports **Joint Multi-Band Spectral Modeling** (`src/spectral_fusion.py`) for cross-band correlation learning
6. Generates training loss & metric curves (PSNR, SSIM)
7. Runs 4× inference (`fast`, `balanced`, or `quality` preset) with multi-band TTA uncertainty heatmaps
8. Visualizes original vs bicubic vs 4× AI super-resolution detail side-by-side
9. Launches interactive Streamlit web dashboard via localtunnel
10. Downloads fine-tuned weights (`model_finetuned_best.pth`) and GeoTIFF outputs

> **Recommended runtime:** Runtime → Change runtime type → **T4 GPU**

## Cell 1 — GPU Verification
Confirms CUDA is available and prints GPU model & total VRAM (T4 ~15 GB).

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Model:', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f'Total VRAM: {props.total_memory / 1024**3:.2f} GB')
else:
    raise RuntimeError('No GPU detected. Go to Runtime > Change runtime type and select T4 GPU.')

## Cell 2 — Install Dependencies & Fix TorchVision Compatibility
Installs rasterio, scikit-image, opencv, and applies torchvision functional_tensor shim.

In [ ]:
# 1. Install scientific & GIS packages
!pip install -q rasterio scikit-image tqdm opencv-python matplotlib

# 2. Create torchvision compatibility shim for functional_tensor
import torchvision, os
tv_dir = os.path.dirname(torchvision.__file__)
ft_path = os.path.join(tv_dir, 'transforms', 'functional_tensor.py')
with open(ft_path, 'w') as f:
    f.write('from torchvision.transforms.functional import *\n')

# 3. Install basicsr & realesrgan
!pip install -q --no-build-isolation basicsr || true
!pip install -q realesrgan || true
print('✓ Dependencies installed successfully.')

## Cell 3 — Clone / Update Repository
Clones your GitHub repository into Colab. If private, uses a Personal Access Token from Colab Secrets.

In [ ]:
import os, sys

# Option A: Use Colab Secrets for private repositories
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    REPO_URL = f'https://{GITHUB_TOKEN}@github.com/AjayBora002/Depth-Wizard.git'
    print('Using GitHub token from Colab Secrets')
except (ImportError, Exception):
    # Option B: Public repo URL fallback
    REPO_URL = 'https://github.com/AjayBora002/Depth-Wizard.git'
    print('Using standard repository URL')

CLONE_DIR = '/content/Depth-Wizard'
PROJECT_DIR = f'{CLONE_DIR}/srm-project'

if not os.path.exists(CLONE_DIR):
    !git clone {REPO_URL} {CLONE_DIR}
else:
    print('Repo already cloned, pulling latest commits...')
    !git -C {CLONE_DIR} pull

# Clear cached modules so latest git updates take effect
for mod in [m for m in list(sys.modules.keys()) if m.startswith('src')]:
    del sys.modules[mod]

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
!ls -la src/

## Cell 4 — Pretrained Baseline Weights
Downloads `RealESRGAN_x4plus.pth` into `src/checkpoints/` if not already cached.

In [ ]:
from src.model import _download_weights
weight_path = _download_weights('x4plus')
print('✓ Weights ready at:', weight_path)

## Cell 5 — Prepare Training Pairs from Real Sentinel-2 Satellite Tiles

Discovers **all real Sentinel-2 satellite tiles** in `data/raw/` (excluding any dummy samples). Tiles the real satellite acquisitions into high-resolution ground truth patches, applies the Sentinel-2 sensor PSF degradation model (downscaled by 4×) to create matched (LR, HR) training pairs directly from your authentic multi-temporal imagery.


In [ ]:
import os
from pathlib import Path
from collections import Counter

# Set to True to rebuild training pairs from your real satellite tiles in data/raw/
REBUILD_TRAINING_PAIRS = False
MAX_PATCHES_PER_TILE = 50   # Balances patches across multi-temporal acquisition dates
PAIRS_DIR = Path('data/training_pairs')

lr_dir = PAIRS_DIR / 'lr'
hr_dir = PAIRS_DIR / 'hr'

# Discover real GeoTIFF tiles (exclude any dummy / sample files)
raw_tiles = sorted([
    p for p in Path('data/raw').rglob('*')
    if p.is_file() and p.suffix.lower() in ('.tif', '.tiff') and 'sample' not in p.name.lower() and 'dummy' not in p.name.lower()
])

print(f'Discovered {len(raw_tiles)} real Sentinel-2 satellite tile(s) in data/raw/:')
for i, t in enumerate(raw_tiles[:10]):
    print(f'  [{i+1}] {t.name}')
if len(raw_tiles) > 10:
    print(f'  ... and {len(raw_tiles) - 10} more real tile(s)')

n_lr = len(list(lr_dir.glob('*.npy'))) if lr_dir.exists() else 0
n_hr = len(list(hr_dir.glob('*.npy'))) if hr_dir.exists() else 0

if n_lr == 0 or n_hr == 0 or REBUILD_TRAINING_PAIRS:
    if not raw_tiles:
        raise FileNotFoundError('No real satellite tiles found in data/raw/. Upload your Sentinel-2 tiles to data/raw/ via the Colab Files panel.')
    
    print(f'\nGenerating training pairs from {len(raw_tiles)} real satellite tile(s)...')
    from src.pair_generation import generate_all_pairs
    generate_all_pairs(
        raw_dir='data/raw',
        out_dir=str(PAIRS_DIR),
        patch_size=512,
        overlap=64,
        scale=4,
        rgb_only=True,
        max_patches_per_tile=MAX_PATCHES_PER_TILE,
    )
    print('✓ Training pairs prepared successfully from real satellite data!')

# Display summary breakdown
lrs = list(lr_dir.glob('*.npy')) if lr_dir.exists() else []
hrs = list(hr_dir.glob('*.npy')) if hr_dir.exists() else []
print(f'\n✓ Total real training patches: {len(lrs)} (LR) / {len(hrs)} (HR)')
if lrs:
    stems = [p.name.split('_00')[0] for p in lrs]
    counts = Counter(stems)
    print('Patches by satellite acquisition source:')
    for stem, cnt in counts.most_common(10):
        print(f'  - {stem}: {cnt} patches')
    if len(counts) > 10:
        print(f'  ... and {len(counts) - 10} other acquisition tiles')


## Cell 6 — Fine-Tune Model with Unified Training Engine (`src/train.py`)
Runs full multi-loss training with **Warmup + Cosine Annealing LR**, **Gradient Checkpointing**, and **Multi-Scale Pyramid Loss**.

**T4 GPU Safe Defaults (~15 GB VRAM):**
- `batch_size = 4`
- `crop_size = 128`
- `use_gradient_checkpointing = True` (saves ~40-50% VRAM)
- `use_amp = True` (FP16 automatic mixed precision for ~2× training speedup)
- Optional: set `JOINT_SPECTRAL = True` to enable joint multi-band cross-spectral attention modeling!

In [ ]:
import sys, torch, logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

# Clear cached modules and free GPU memory
for mod in [m for m in list(sys.modules.keys()) if m.startswith('src')]:
    del sys.modules[mod]
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB allocated, {torch.cuda.memory_reserved()/1024**3:.2f} GB reserved')

from src.train import train

# Set JOINT_SPECTRAL = True to enable joint multi-band cross-attention modeling
JOINT_SPECTRAL = False

results = train(
    pairs_dir='data/training_pairs',
    output_dir='src/checkpoints',
    model_key='x4plus',
    epochs=50,
    batch_size=4,
    crop_size=128,
    lr=5e-5,
    warmup_epochs=5,
    lambda_l1=1.0,
    lambda_perceptual=0.10,     # VGG16 feature loss
    lambda_sam=0.05,            # Spectral Angle Mapper (reflectance consistency)
    lambda_edge=0.05,           # Sobel gradient loss (linear features/roads)
    lambda_freq=0.02,           # 2D FFT loss (micro-texture details)
    lambda_multiscale=0.05,     # Multi-scale pyramid loss
    joint_spectral=JOINT_SPECTRAL,
    save_every=5,
    num_workers=2,
    use_amp=True,               # FP16 mixed precision
    use_gradient_checkpointing=True,
    early_stopping_patience=15,
    time_budget_hours=3.0,
    ablation_name='colab_joint_spectral' if JOINT_SPECTRAL else 'colab_full',
)

print('\n' + '=' * 60)
print('✓ Training complete!')
print('Best checkpoint:', results['best_checkpoint'])
hist = results['history']
print(f'Epochs completed: {len(hist["train_loss"])}')
print(f'Best Validation PSNR: {max(hist["val_psnr"]):.2f} dB')
if 'val_ssim' in hist and hist['val_ssim']:
    print(f'Best Validation SSIM: {max(hist["val_ssim"]):.4f}')
print(f'Total training time: {sum(hist["epoch_time"])/60:.1f} minutes')
print('=' * 60)

## Cell 6b (Optional) — Run Controlled Ablation Experiments
Runs controlled comparisons using `scripts/run_ablations.py` to produce benchmark numbers for your SIH report.

In [ ]:
# Example: Run SAM loss ablation alone, or run the entire suite:
# !python scripts/run_ablations.py --ablation with_sam --epochs 25
# !python scripts/run_ablations.py --all --epochs 25 --output-root data/ablations/

## Cell 7 — Plot Training & Validation Curves
Plots loss convergence, PSNR improvement, SSIM, and epoch duration.

In [ ]:
import json
import matplotlib.pyplot as plt

with open('src/checkpoints/training_history.json') as f:
    hist = json.load(f)

epochs = range(1, len(hist['train_loss']) + 1)
has_ssim = 'val_ssim' in hist and len(hist['val_ssim']) == len(hist['train_loss'])

ncols = 4 if has_ssim else 3
fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4))

axes[0].plot(epochs, hist['train_loss'], 'b-o', markersize=4, label='Train Loss')
axes[0].plot(epochs, hist['val_loss'], 'r-o', markersize=4, label='Val Loss')
axes[0].set_title('Composite Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, hist['val_psnr'], 'g-o', markersize=4)
axes[1].set_title('Validation PSNR (dB)')
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)

metric_axis = 2
if has_ssim:
    axes[2].plot(epochs, hist['val_ssim'], 'c-o', markersize=4)
    axes[2].set_title('Validation SSIM')
    axes[2].set_xlabel('Epoch')
    axes[2].grid(True, alpha=0.3)
    metric_axis = 3

axes[metric_axis].plot(epochs, hist['epoch_time'], 'm-o', markersize=4)
axes[metric_axis].set_title('Epoch Time (s)')
axes[metric_axis].set_xlabel('Epoch')
axes[metric_axis].grid(True, alpha=0.3)

plt.suptitle('SIH26142 — Sentinel-2 SR Fine-Tuning Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('src/checkpoints/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Curves saved to src/checkpoints/training_curves.png')

## Cell 8 — Run 4× Super-Resolution Inference (GPU Accelerated)
Executes full inference on a Sentinel-2 tile with **FP16**, **Hann-window blending**, and **TTA uncertainty**.

Presets:
- `fast`: 512px patches, no TTA (ultra-fast)
- `balanced`: 512px patches, 4× TTA uncertainty (recommended default)
- `quality`: 384px patches, 8× TTA uncertainty + enhanced sharpening

In [ ]:
import os, sys
from pathlib import Path

# Choose preset: 'fast', 'balanced', or 'quality'
INFERENCE_PRESET = 'balanced'
USE_JOINT_SPECTRAL = False

# 1. Dynamically select real Sentinel-2 input tile (exclude dummy / sample files)
raw_tiles = sorted([
    p for p in Path('data/raw').rglob('*')
    if p.is_file() and p.suffix.lower() in ('.tif', '.tiff') and 'sample' not in p.name.lower() and 'dummy' not in p.name.lower()
])

# Prefer true-color RGB tiles for visualization if available
true_color_tiles = [p for p in raw_tiles if 'true_color' in p.name.lower()]
if true_color_tiles:
    input_tile = str(true_color_tiles[0])
elif raw_tiles:
    input_tile = str(raw_tiles[0])
else:
    raise FileNotFoundError('No real .tif / .tiff tiles found in data/raw/. Upload real satellite tiles first.')

input_stem = Path(input_tile).stem
print(f'Selected real Sentinel-2 input tile: {input_tile}')

# 2. Find best checkpoint
best_ckpt = Path('src/checkpoints/model_finetuned_best.pth')
base_ckpt = Path('src/checkpoints/RealESRGAN_x4plus.pth')
ckpt_path = str(best_ckpt if best_ckpt.exists() else base_ckpt)
print(f'Using checkpoint: {ckpt_path}')

# 3. Run inference on real satellite tile
from src.inference import run_inference

PRESETS = {
    'fast':     dict(patch_size=512, overlap=48, compute_uncertainty=False, tta_n=0, sharpen=0.4, high_quality=False),
    'balanced': dict(patch_size=512, overlap=64, compute_uncertainty=True,  tta_n=4, sharpen=0.5, high_quality=False),
    'quality':  dict(patch_size=384, overlap=64, compute_uncertainty=True,  tta_n=8, sharpen=0.6, high_quality=True),
}
settings = PRESETS[INFERENCE_PRESET]
output_file = f'data/outputs/sr_{input_stem}_{INFERENCE_PRESET}.tif'

result = run_inference(
    input_path=input_tile,
    output_path=output_file,
    checkpoint_path=ckpt_path,
    half=True,
    tile_size=256,
    joint_spectral=USE_JOINT_SPECTRAL,
    **settings,
)

print('\n' + '=' * 60)
print('✓ 4× Super-Resolved GeoTIFF:', result['sr_output'])
print('✓ Uncertainty Heatmap:', result['uncertainty_map'])
print('=' * 60)


## Cell 9 — Full-Scene & Zoomed-In Comparison Visualizer
Renders the input 10m tile, 4× Bicubic baseline, 4× AI Super-Resolution, and the TTA Uncertainty Heatmap.

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

def norm_rgb(arr):
    rgb = arr[:3].transpose(1, 2, 0).astype(np.float32)
    p2, p98 = np.percentile(rgb, (2, 98))
    if p98 > p2:
        rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)
    else:
        rgb = np.clip(rgb / 255.0, 0, 1)
    return rgb

lr_file = Path(input_tile)
sr_file = Path(result['sr_output'])

with rasterio.open(lr_file) as f_lr:
    lr_rgb = norm_rgb(f_lr.read())

with rasterio.open(sr_file) as f_sr:
    sr_rgb = norm_rgb(f_sr.read())

unc_path = Path(result['uncertainty_map'])
has_unc = unc_path.exists()
if has_unc:
    unc_img = plt.imread(str(unc_path))

# Center crop for detail inspection
r0, r1 = int(lr_rgb.shape[0] * 0.40), int(lr_rgb.shape[0] * 0.55)
c0, c1 = int(lr_rgb.shape[1] * 0.40), int(lr_rgb.shape[1] * 0.55)
lr_crop = lr_rgb[r0:r1, c0:c1]
scale_y = sr_rgb.shape[0] / lr_rgb.shape[0]
scale_x = sr_rgb.shape[1] / lr_rgb.shape[1]
sr_crop = sr_rgb[int(r0*scale_y):int(r1*scale_y), int(c0*scale_x):int(c1*scale_x)]
bicubic_crop = cv2.resize(lr_crop, (sr_crop.shape[1], sr_crop.shape[0]), interpolation=cv2.INTER_CUBIC)

fig = plt.figure(figsize=(18, 11))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.2])

ax0 = fig.add_subplot(gs[0, 0])
ax0.imshow(lr_rgb)
ax0.set_title(f'Low-Res Input: Sentinel-2 10m ({lr_rgb.shape[1]}x{lr_rgb.shape[0]})', fontsize=12)
ax0.axis('off')

ax1 = fig.add_subplot(gs[0, 1])
ax1.imshow(sr_rgb)
ax1.set_title(f'4× Super-Resolved: 2.5m Pitch ({sr_rgb.shape[1]}x{sr_rgb.shape[0]})', fontsize=12, fontweight='bold', color='darkgreen')
ax1.axis('off')

ax2 = fig.add_subplot(gs[0, 2])
if has_unc:
    ax2.imshow(unc_img)
    ax2.set_title('TTA Uncertainty Heatmap (Confidence)', fontsize=12)
else:
    ax2.text(0.5, 0.5, 'Uncertainty Disabled', ha='center', va='center')
ax2.axis('off')

# Row 2: Zoomed-in comparisons
ax3 = fig.add_subplot(gs[1, 0])
ax3.imshow(lr_crop, interpolation='nearest')
ax3.set_title('Zoomed-In: Raw 10m Pixels (Blocky)', fontsize=12, fontweight='bold', color='red')
ax3.axis('off')

ax4 = fig.add_subplot(gs[1, 1])
ax4.imshow(bicubic_crop)
ax4.set_title('Zoomed-In: Standard 4× Bicubic (Blurry)', fontsize=12, fontweight='bold', color='darkorange')
ax4.axis('off')

ax5 = fig.add_subplot(gs[1, 2])
ax5.imshow(sr_crop)
ax5.set_title('Zoomed-In: 4× AI Super-Resolved (Crisp)', fontsize=12, fontweight='bold', color='darkgreen')
ax5.axis('off')

plt.suptitle('SIH26142 Sentinel-2 Super-Resolution: Full Scene & Detail Verification', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('data/outputs/sr_comparison.png', dpi=200, bbox_inches='tight')
plt.savefig(f'data/outputs/sr_{input_stem}_comparison.png', dpi=200, bbox_inches='tight')
plt.show()


## Cell 10 — Launch Streamlit Web Dashboard (GPU Accelerated via localtunnel)
Runs `dashboard/app.py` in the background and opens a public tunnel URL.

In [ ]:
!pip install -q streamlit streamlit-folium streamlit-image-comparison pydeck plotly
!npm install -g localtunnel -q

import urllib.request
public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print('═' * 60)
print(f'YOUR TUNNEL PASSWORD (ENDPOINT IP): {public_ip}')
print('═' * 60)

!streamlit run dashboard/app.py --server.port 8501 --server.headless true & npx localtunnel --port 8501

## Cell 11 — Download Checkpoints & Outputs
Downloads the best fine-tuned model checkpoint and SR outputs directly to your local computer.

In [ ]:
from google.colab import files
import os
from pathlib import Path

to_download = [
    'src/checkpoints/model_finetuned_best.pth',
    'src/checkpoints/model_finetuned_final.pth',
    'src/checkpoints/training_history.json',
    'src/checkpoints/training_curves.png',
    'data/outputs/sr_comparison.png',
]
to_download.extend(str(p) for p in sorted(Path('data/outputs').glob('*.tif')))
to_download.extend(str(p) for p in sorted(Path('data/outputs').glob('*.png')))

downloaded = set()
for path in to_download:
    if path in downloaded:
        continue
    downloaded.add(path)
    if os.path.exists(path):
        print(f'Downloading: {path}')
        try:
            files.download(path)
        except Exception as e:
            print(f'Could not download {path}: {e}')
    else:
        print(f'Skipped (not found): {path}')

print('✓ Done!')